# ValleyFever-NewsCast

## 4. Regression Analysis

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [1]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/ValleyFever-NewsCast/'):
        !git clone https://github.com/Adrian1840/ValleyFever-NewsCast
    os.chdir('/content/ValleyFever-NewsCast')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Cloning into 'ValleyFever-NewsCast'...
remote: Enumerating objects: 465, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 465 (delta 46), reused 1 (delta 1), pack-reused 368 (from 3)
Receiving objects: 100% (465/465), 2.66 MiB | 12.97 MiB/s, done.
Resolving deltas: 100% (184/184), done.
Current working directory: /content/ValleyFever-NewsCast


## Import Libraries and Data Loading

In [2]:
  # Basic Packages
  import numpy as np
  import pandas as pd
  import statsmodels.formula.api as smf #Linear Models Module

In [3]:
model_df_pth = "/content/ValleyFever-NewsCast/data/processed/final_model_dataframe.csv"
final_model_dataframe = pd.read_csv(model_df_pth)
final_model_dataframe.head(5)

,Log_Rate,Num_Articles_lag3,cv_mentions_rate_lag3,risk_mentions_rate_lag3,County,Season,Year-Month
0,1.968000,0,0.0,0.0,Fresno,Fall,2008-10
1,1.483416,1,0.0,3.0,Fresno,Fall,2008-11
2,2.013055,2,0.0,2.5,Fresno,Winter,2008-12
3,2.030750,0,0.0,0.0,Fresno,Winter,2009-01
4,1.909784,2,0.0,0.0,Fresno,Winter,2009-02


# Regression Model (Before Diagnostics)

$$\text{Log_Rate}_t \sim  \text{CV_mentions_rate}_{t-3} + \text{Risk_mentions_rate}_{t-3} + \text{Num_Articles}_{t-3} * \text{C(County) + C(Season)}$$

In [4]:
multi_mod = smf.ols("Log_Rate ~ risk_mentions_rate_lag3 + cv_mentions_rate_lag3 + Num_Articles_lag3 * C(County) + C(Season) ", data=final_model_dataframe).fit()
print(multi_mod.summary())

                            OLS Regression Results                            
Dep. Variable:               Log_Rate   R-squared:                       0.634
Model:                            OLS   Adj. R-squared:                  0.616
Method:                 Least Squares   F-statistic:                     35.73
Date:                Sat, 30 May 2026   Prob (F-statistic):           2.52e-32
Time:                        00:27:52   Log-Likelihood:                -113.23
No. Observations:                 174   AIC:                             244.5
Df Residuals:                     165   BIC:                             272.9
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

## Influence Diagnostics

### Calculating Cook's Distance and DFFITS of each point

In [5]:
influence = multi_mod.get_influence()

cooks_d, _ = influence.cooks_distance
dffits, _ = influence.dffits

final_model_dataframe["cooks_d"] = cooks_d
final_model_dataframe["dffits"] = dffits

In [6]:
n = len(final_model_dataframe)
k = int(multi_mod.df_model)  #gets number of predictors in model


threshold_cook = 4 / n
threshold_dffits = 2 * np.sqrt((k + 2) / (n - k - 2))

## Cook's Distance Model

In [7]:
cooks_df = final_model_dataframe[final_model_dataframe['cooks_d'] <= threshold_cook].copy()


$$\text{Log_Rate}_t \sim  \text{Cen_Valley_mentions}_{t-3} + \text{Risk_Word_mentions}_{t-3} + \text{Num_Articles}_{t-3} * \text{C(County) + C(Season)}$$

In [8]:
cooks_mod_clean = smf.ols("""
Log_Rate ~ risk_mentions_rate_lag3
         + cv_mentions_rate_lag3
         + Num_Articles_lag3 * C(County)
         + C(Season)
""", data=cooks_df).fit(cov_type="cluster", cov_kwds={"groups": cooks_df["Year-Month"]})

print(cooks_mod_clean.summary())

                            OLS Regression Results                            
Dep. Variable:               Log_Rate   R-squared:                       0.661
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     98.82
Date:                Sat, 30 May 2026   Prob (F-statistic):           4.94e-40
Time:                        00:27:57   Log-Likelihood:                -101.77
No. Observations:                 168   AIC:                             221.5
Df Residuals:                     159   BIC:                             249.7
Df Model:                           8                                         
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [13]:
cooks_model = pd.DataFrame({
    "Variable": cooks_mod_clean.params.index,
    "Coefficient": cooks_mod_clean.params.values,
    "Std_Error": cooks_mod_clean.bse.values,
    "P_Value": cooks_mod_clean.pvalues.values
})

cooks_model.to_csv(
    "results/regression/cooks_model.csv",
    index=False
)

## Difference of Fits Model

In [11]:
dffits_df = final_model_dataframe[final_model_dataframe["dffits"] <= threshold_dffits].copy()

In [12]:
dffits_mod_clean = smf.ols("""
Log_Rate ~ risk_mentions_rate_lag3
         + cv_mentions_rate_lag3
         + Num_Articles_lag3 * C(County)
         + C(Season)
""", data=dffits_df).fit(cov_type="cluster", cov_kwds={"groups": dffits_df["Year-Month"]})

print(dffits_mod_clean.summary())

                            OLS Regression Results                            
Dep. Variable:               Log_Rate   R-squared:                       0.641
Model:                            OLS   Adj. R-squared:                  0.623
Method:                 Least Squares   F-statistic:                     87.02
Date:                Sat, 30 May 2026   Prob (F-statistic):           6.39e-38
Time:                        00:29:07   Log-Likelihood:                -111.32
No. Observations:                 173   AIC:                             240.6
Df Residuals:                     164   BIC:                             269.0
Df Model:                           8                                         
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [15]:
dffits_model = pd.DataFrame({
    "Variable": dffits_mod_clean.params.index,
    "Coefficient": dffits_mod_clean.params.values,
    "Std_Error": dffits_mod_clean.bse.values,
    "P_Value": dffits_mod_clean.pvalues.values
})

dffits_model.to_csv(
    "results/regression/dffits_model.csv",
    index=False
)